# Laboratorio 7 - Task 3

In [ ]:
import os
import zipfile
import glob
from sklearn.model_selection import train_test_split
import shutil
from collections import Counter
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from torchvision import models


## 1. Preparación del dataset

### a. Descarga

In [ ]:
os.makedirs('/root/.kaggle', exist_ok=True)

In [ ]:
import json

kaggle_dict = {
    "username": "arielamishaan",
    "key": "KGAT_5e7b66abdc34db9c14fa67d2623b63ee"
}

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_dict, f)

os.chmod("/root/.kaggle/kaggle.json", 600)

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [02:21<00:00, 17.4MB/s]



In [ ]:

with zipfile.ZipFile('chest-xray-pneumonia.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/data')

### b. División con estratificación

In [ ]:
data_dir = '/content/data/chest_xray/train'

normal = glob.glob(data_dir + '/NORMAL/*.jpeg')
pneumonia = glob.glob(data_dir + '/PNEUMONIA/*.jpeg')

all_images = normal + pneumonia
labels = [0]*len(normal) + [1]*len(pneumonia)

In [ ]:
# split tarin (70%) y temp (30%)
train_imgs, temp_imgs, train_labels, temp_labels = train_test_split(
    all_images, labels,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

# split val (15%) y test (15%)
val_imgs, test_imgs, val_labels, test_labels = train_test_split(
    temp_imgs, temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)


In [ ]:
# estructura final
def create_dirs(base):
    for split in ['train', 'val', 'test']:
        for cls in ['NORMAL', 'PNEUMONIA']:
            os.makedirs(os.path.join(base, split, cls), exist_ok=True)

create_dirs('/content/dataset')

In [ ]:
# Crear imágenes
def copy_images(imgs, labels, split):
    for img, label in zip(imgs, labels):
        cls = 'NORMAL' if label == 0 else 'PNEUMONIA'
        shutil.copy(img, f'/content/dataset/{split}/{cls}/')

copy_images(train_imgs, train_labels, 'train')
copy_images(val_imgs, val_labels, 'val')
copy_images(test_imgs, test_labels, 'test')

In [ ]:
# verificación
print("Train:", Counter(train_labels))
print("Val:", Counter(val_labels))
print("Test:", Counter(test_labels))


Train: Counter({1: 2712, 0: 939})
Val: Counter({1: 581, 0: 201})
Test: Counter({1: 582, 0: 201})


Se dividió el dataset en 70% entrenamiento, 15% validación y 15% prueba utilizando muestreo estratificado (stratified sampling). Esta técnica asegura que la proporción de imágenes normales y con neumonía se mantenga consistente en cada subconjunto.

Esto es especialmente importante en el contexto médico, donde existe un desbalance de clases, ya que una división aleatoria sin estratificación podría generar conjuntos con distribuciones sesgadas. Esto afectaría tanto el entrenamiento del modelo como la evaluación de su desempeño, produciendo métricas poco representativas del entorno real.

### c. Data Augmentation

In [ ]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),

    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),

    transforms.ColorJitter(brightness=0.1, contrast=0.1),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
# Data Loaders

data_dir = '/content/dataset'

train_dataset = ImageFolder(f"{data_dir}/train", transform=train_transforms)
val_dataset = ImageFolder(f"{data_dir}/val", transform=val_test_transforms)
test_dataset = ImageFolder(f"{data_dir}/test", transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



Se implementó Data Augmentation únicamente en el conjunto de entrenamiento con el objetivo de mejorar la capacidad de generalización del modelo sin incrementar artificialmente el tamaño del dataset con datos irreales.

Las transformaciones aplicadas incluyen:


*   **Rotaciones leves (±10°)**: simulan pequeñas variaciones en la posición del paciente durante la adquisición de la radiografía.
*   **Flip horizontal**: representa posibles cambios en la orientación de la imagen sin alterar la anatomía global relevante.
* **Random Resized Crop**: simula variaciones en el encuadre o zoom del equipo de rayos X.
* **Ajustes de brillo y contraste**: modelan diferencias en condiciones de exposición y calibración del equipo.

Estas transformaciones son válidas porque no alteran las estructuras anatómicas clave necesarias para el diagnóstico, como opacidades pulmonares.

Para los conjuntos de validación y prueba no se aplicaron transformaciones aleatorias, únicamente normalización. Esto es fundamental, ya que permite evaluar el modelo sobre datos no modificados, obteniendo métricas que reflejan su desempeño real en un entorno clínico.

## 2. Modelos ImageNet

### a. Congelar capas y modificar cabezal

#### ResNet18

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cargar modelo preentrenado
resnet = resnet18(weights=ResNet18_Weights.DEFAULT)

# Congelar capas base
for param in resnet.parameters():
    param.requires_grad = False

# Reemplazar cabezal
num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, 2)

# Asegurar que el cabezal es entrenable
for param in resnet.fc.parameters():
    param.requires_grad = True

resnet = resnet.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 144MB/s]


Se congelaron las capas convolucionales del modelo preentrenado para preservar las características visuales generales aprendidas en ImageNet, como bordes y texturas. Estas representaciones son transferibles incluso a imágenes médicas.

Únicamente se entrenó el cabezal de clasificación, adaptándolo al problema binario, lo cual reduce el riesgo de sobreajuste dado el tamaño limitado del dataset y disminuye el costo computacional del entrenamiento.

#### MobileNetV2

In [ ]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

# Congelar capas base
for param in mobilenet.parameters():
    param.requires_grad = False

# Reemplazar clasificador
num_features = mobilenet.classifier[1].in_features
mobilenet.classifier[1] = nn.Linear(num_features, 2)

# Asegurar entrenamiento del clasificador
for param in mobilenet.classifier.parameters():
    param.requires_grad = True

mobilenet = mobilenet.to(device)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 221MB/s]


En el caso de MobileNetV2, se congeló el extractor de características (features) y se reemplazó el clasificador final (classifier) para adaptarlo al problema binario. Esta arquitectura está optimizada para eficiencia computacional, lo que la hace especialmente adecuada para entornos con recursos limitados, como clínicas sin GPU dedicada.

A diferencia de ResNet18, MobileNetV2 utiliza convoluciones separables en profundidad (depthwise separable convolutions), lo que reduce significativamente el número de parámetros y el costo computacional, manteniendo un buen desempeño.

### b. Entrenamiento (misma loss + optimizador) con early stopping

In [ ]:
# Definir loss y optimizador
criterion = nn.CrossEntropyLoss()

optimizer_resnet = torch.optim.Adam(resnet.fc.parameters(), lr=1e-3)
optimizer_mobilenet = torch.optim.Adam(mobilenet.classifier[1].parameters(), lr=1e-3)

# Early Stopping mejorado
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.best_loss = float('inf')
        self.counter = 0

    def step(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

# Función de entrenamiento mejorada
def train_model(model, train_loader, val_loader, optimizer, epochs=10, name="Model"):

    early_stopping = EarlyStopping(patience=3)

    train_losses = []
    val_losses = []

    best_model_wts = model.state_dict()
    best_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)

        # Validación
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        # Logs
        print(f"[{name}] Epoch {epoch+1}/{epochs}")
        print(f"   Train Loss: {train_loss:.4f}")
        print(f"   Val Loss:   {val_loss:.4f}")

        # Guardar mejor modelo
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_wts = model.state_dict()

        # Early stopping
        if early_stopping.step(val_loss):
            print(f"Early stopping activado en época {epoch+1}")
            break

    # Restaurar mejor modelo
    model.load_state_dict(best_model_wts)

    return train_losses, val_losses

In [ ]:
# entrenar resnet
train_model(resnet, train_loader, val_loader, optimizer_resnet, epochs=15, name="ResNet18")


[ResNet18] Epoch 1/15
   Train Loss: 0.3551
   Val Loss:   0.3151
[ResNet18] Epoch 2/15
   Train Loss: 0.2198
   Val Loss:   0.1464
[ResNet18] Epoch 3/15
   Train Loss: 0.2118
   Val Loss:   0.2290
[ResNet18] Epoch 4/15
   Train Loss: 0.1906
   Val Loss:   0.2099
[ResNet18] Epoch 5/15
   Train Loss: 0.1652
   Val Loss:   0.1784
Early stopping activado en época 5


([0.3551325509081716,
  0.21976726696543072,
  0.21179973534915758,
  0.19062692824265232,
  0.16516803862607998],
 [0.3151232647895813,
  0.14635920085012913,
  0.22902426183223723,
  0.2099357134103775,
  0.17837866827845572])

In [ ]:
# entrenar mobilenet
train_model(mobilenet, train_loader, val_loader, optimizer_mobilenet, epochs=15, name="MobileNetV2")


[MobileNetV2] Epoch 1/15
   Train Loss: 0.3621
   Val Loss:   0.2593
[MobileNetV2] Epoch 2/15
   Train Loss: 0.2402
   Val Loss:   0.2270
[MobileNetV2] Epoch 3/15
   Train Loss: 0.2070
   Val Loss:   0.1984
[MobileNetV2] Epoch 4/15
   Train Loss: 0.2038
   Val Loss:   0.1813
[MobileNetV2] Epoch 5/15
   Train Loss: 0.1887
   Val Loss:   0.1566
[MobileNetV2] Epoch 6/15
   Train Loss: 0.1883
   Val Loss:   0.1741
[MobileNetV2] Epoch 7/15
   Train Loss: 0.1735
   Val Loss:   0.1536
[MobileNetV2] Epoch 8/15
   Train Loss: 0.1649
   Val Loss:   0.1807
[MobileNetV2] Epoch 9/15
   Train Loss: 0.1775
   Val Loss:   0.1669
[MobileNetV2] Epoch 10/15
   Train Loss: 0.1755
   Val Loss:   0.1476
[MobileNetV2] Epoch 11/15
   Train Loss: 0.1656
   Val Loss:   0.1809
[MobileNetV2] Epoch 12/15
   Train Loss: 0.1603
   Val Loss:   0.2118
[MobileNetV2] Epoch 13/15
   Train Loss: 0.1580
   Val Loss:   0.1530
Early stopping activado en época 13


([0.36206498444080354,
  0.24018173612978147,
  0.20697141639564348,
  0.2038478395861128,
  0.18869282606503238,
  0.18829838244811348,
  0.17349921118306078,
  0.1649435926714669,
  0.1774643665422564,
  0.17546895522138348,
  0.16564763029632362,
  0.16034546272586223,
  0.15801269450913305],
 [0.25929262399673464,
  0.227029972076416,
  0.19840254098176957,
  0.18131343096494676,
  0.15662344180047513,
  0.17409379974007608,
  0.15362178817391395,
  0.18072593152523042,
  0.1668934069573879,
  0.14756452545523643,
  0.18088243156671524,
  0.2117601777613163,
  0.15297407239675523])

Durante el entrenamiento, ambos modelos mostraron una reducción progresiva en la pérdida de entrenamiento, lo que indica que lograron aprender patrones relevantes del dataset.

En el caso de ResNet18, se observó que la pérdida de validación alcanzó su valor mínimo en las primeras épocas (alrededor de la época 2), pero posteriormente comenzó a aumentar, evidenciando un comportamiento de sobreajuste. El Early Stopping permitió detener el entrenamiento oportunamente en la época 5.

Por otro lado, MobileNetV2 presentó una disminución más gradual y estable en la pérdida de validación, alcanzando su mejor desempeño alrededor de la época 10. Este comportamiento sugiere una mejor capacidad de generalización y menor tendencia al sobreajuste temprano.

En conjunto, estos resultados indican que, aunque ambos modelos logran aprender la tarea, MobileNetV2 presenta un entrenamiento más estable y potencialmente más adecuado para escenarios reales con datos limitados.

## 3. Métricas

### Accuracy - F1-Score - Recall

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score
import time
import os

#Función de evaluación
def evaluate_model(model, loader):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)  # clase positiva = neumonía
    recall = recall_score(y_true, y_pred)

    return acc, f1, recall

#Evaluación de Ambos modelos
resnet_acc, resnet_f1, resnet_recall = evaluate_model(resnet, test_loader)
mobilenet_acc, mobilenet_f1, mobilenet_recall = evaluate_model(mobilenet, test_loader)

print("ResNet18 -> Accuracy:", resnet_acc, "F1:", resnet_f1, "Recall:", resnet_recall)
print("MobileNetV2 -> Accuracy:", mobilenet_acc, "F1:", mobilenet_f1, "Recall:", mobilenet_recall)

ResNet18 -> Accuracy: 0.933588761174968 F1: 0.9540636042402827 Recall: 0.9278350515463918
MobileNetV2 -> Accuracy: 0.9425287356321839 F1: 0.9606299212598425 Recall: 0.9432989690721649


### Tamaño del Modelo

In [ ]:
# Guardar modelos
torch.save(resnet.state_dict(), "resnet.pth")
torch.save(mobilenet.state_dict(), "mobilenet.pth")

# Calcular tamaño
resnet_size = os.path.getsize("resnet.pth") / (1024 * 1024)
mobilenet_size = os.path.getsize("mobilenet.pth") / (1024 * 1024)

print("ResNet size (MB):", resnet_size)
print("MobileNet size (MB):", mobilenet_size)

ResNet size (MB): 42.70930004119873
MobileNet size (MB): 8.72712230682373


### Tiempo de inferencia

In [ ]:
def measure_inference_time(model, loader, num_batches=100):
    model.eval()
    times = []

    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break

            images = images.to(device)

            start = time.time()
            _ = model(images)
            end = time.time()

            times.append(end - start)

    return (sum(times) / len(times)) * 1000  # ms

resnet_time = measure_inference_time(resnet, test_loader)
mobilenet_time = measure_inference_time(mobilenet, test_loader)

print("ResNet time (ms):", resnet_time)
print("MobileNet time (ms):", mobilenet_time)

ResNet time (ms): 4.03325080871582
MobileNet time (ms): 6.064119338989258


Al comparar ambos modelos, se observa que MobileNetV2 supera a ResNet18 en todas las métricas clave. En particular, presenta mayor Accuracy (0.9425 vs 0.9336), mayor F1-score (0.9606 vs 0.9541) y, más importante, mayor sensibilidad (Recall) para la clase de neumonía (0.9433 vs 0.9278).

Desde una perspectiva clínica, la sensibilidad es la métrica más crítica, ya que mide la capacidad del modelo para detectar correctamente los casos positivos. Un mayor Recall implica una menor probabilidad de falsos negativos, lo cual es fundamental en el diagnóstico médico.

Además, MobileNetV2 presenta una ventaja significativa en términos de tamaño del modelo (8.73 MB frente a 42.71 MB), lo que lo hace mucho más adecuado para su despliegue en entornos con recursos limitados.

Aunque ResNet18 muestra un tiempo de inferencia ligeramente menor, la diferencia es marginal (≈2 ms) y no representa un impacto significativo en un flujo clínico real.

## 4. Reporte
En este trabajo se evaluaron dos modelos preentrenados, ResNet18 y MobileNetV2, adaptados mediante transfer learning para clasificar radiografías de tórax en dos clases: normal y neumonía. Ambos modelos se entrenaron bajo las mismas condiciones (misma función de pérdida, optimizador y estrategia de early stopping), con el objetivo de compararlos de manera justa.

### **1. Comparación de Resultados**

| Modelo| Accuracy | F1-Score | Recall |Tamaño (MB)| Tiempo (ms) |
|---|---|---|---|---|---|
| ResNet50 | 0.9336 | 0.9541 | 0.9278 | 42.71 | 4.03 |
| MobileNetV2 | 0.9425 | 0.9606 | 0.9433 | 8.73 | 6.06 |


### **2. Análisis de desempeño**

Los dos modelos obtuvieron buenos resultados, pero MobileNetV2 destaca ligeramente en todas las métricas. En especial, tiene un mayor recall para la clase de neumonía, lo cual es clave en este tipo de problema. Detectar correctamente a los pacientes enfermos es más importante que simplemente tener un buen accuracy general.

ResNet18 también tiene buen desempeño, pero mostró una tendencia a sobreajustar más rápido durante el entrenamiento, mientras que MobileNetV2 se comportó de forma más estable.

### **3.  Eficiencia y despliegue**

Uno de los objetivos del proyecto es poder usar el modelo en clínicas con recursos limitados. Aquí es donde MobileNetV2 tiene una ventaja clara: su tamaño es mucho menor (8.73 MB frente a 42.71 MB). Esto hace que sea más fácil de almacenar y ejecutar en computadoras sin GPU.

En cuanto al tiempo de inferencia, ResNet18 es un poco más rápido, pero la diferencia es de apenas unos milisegundos, lo cual no es relevante en la práctica. Ambos modelos son suficientemente rápidos para un entorno clínico.

### **4. Relación entre tamaño y desempeño**

Si se compara el tamaño del modelo con su desempeño (especialmente el F1-score), MobileNetV2 resulta más eficiente. Ofrece mejores resultados ocupando mucho menos espacio, lo cual es una ventaja importante en el contexto del proyecto.

### **5. Limitaciones**

A pesar de los buenos resultados, hay varias limitaciones que hay que tener en cuenta:

*   El dataset no es muy grande, lo que puede afectar la capacidad del modelo para generalizar.
* No se probaron datos de diferentes hospitales o equipos de rayos X.
* Es posible que existan sesgos en los datos utilizados.

Por esto, no sería recomendable usar el modelo directamente en producción sin una validación adicional.

### **6. Uso de capas congeladas**

Se decidió congelar las capas base de los modelos y entrenar solo el cabezal de clasificación. Esto ayudó a evitar sobreajuste y a aprovechar las características generales aprendidas en ImageNet. Dado el tamaño del dataset, esta decisión fue adecuada. Sin embargo, con más datos, sería interesante probar fine-tuning más profundo.

### **7. Generalización**

Es probable que el modelo no funcione igual en todos los contextos. Diferencias en equipos, calidad de imagen o incluso en la población pueden afectar el desempeño. Por eso, sería necesario evaluar el modelo en nuevos datos antes de usarlo en otros hospitales.

### **8. Recomendación final**

Tomando todo en cuenta, se recomienda usar MobileNetV2. Tiene mejor desempeño en las métricas más importantes, especialmente en la detección de neumonía, y además es mucho más liviano. Esto lo hace más adecuado para el contexto de clínicas rurales, donde los recursos son limitados.






